# Did "Reorder in 1 Tap" actually work?
### Measuring feature impact without an A/B test — difference-in-differences

**The situation.** QuickKart (a simulated quick-commerce app) shipped a
"Reorder in 1 Tap" feature — one tap repeats your last basket. Product rolled it
out to **5 cities at the start of week 14**; **7 cities** did not get it.
There was **no randomised A/B test**: engineering shipped city by city.

Leadership asks: *"Did it work? Should we roll it out everywhere?"*

**The problem with the obvious answer.** Orders in the treated cities went up
after launch — but orders went up everywhere, because of seasonality and general
growth. Comparing a city to its own past credits the feature with things that
would have happened anyway.

**The method.** Difference-in-differences (DiD):

> **effect = (treated after − treated before) − (control after − control before)**

The second bracket estimates *what would have happened anyway*. Subtracting it
leaves what the feature actually did.

DiD is only valid if the two groups **would have moved in parallel** without the
feature. That assumption is testable, and most of this notebook is spent testing
it rather than computing the estimate — which is the honest ratio of effort.

> ⚠️ All data here is **simulated**. No real company data is used or implied. The
> effects are injected on purpose so the analysis can be scored against a known
> right answer.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from scipy import stats

pd.set_option("display.width", 120)
rng = np.random.default_rng(7)

LAUNCH_WEEK = 14      # first week the feature was live in treated cities
STEADY_FROM = 18      # week 18+ = at least 4 weeks after launch (novelty gone)

### Load the data

Running locally: the files sit in `data/output/`.
Running in **Google Colab**: nothing to change — the notebook reads the CSVs
straight from the public repo (`Sumedh222004/quickkart-feature-impact`).


In [ ]:
import os

BASE = "../data/output"                       # local path (repo layout)
if not os.path.exists(f"{BASE}/city_week_panel.csv"):
    BASE = "data/output"                      # if you run from the repo root
if not os.path.exists(f"{BASE}/city_week_panel.csv"):
    # Colab / anywhere else: read straight from the public GitHub repo
    BASE = ("https://raw.githubusercontent.com/Sumedh222004/"
            "quickkart-feature-impact/main/data/output")

panel = pd.read_csv(f"{BASE}/city_week_panel.csv")
users = pd.read_csv(f"{BASE}/user_period.csv")

print(f"panel: {panel.shape[0]} rows = "
      f"{panel.city.nunique()} cities x {panel.week.nunique()} weeks")
print("treated:", sorted(panel.loc[panel.treated == 1, 'city'].unique()))
print("control:", sorted(panel.loc[panel.treated == 0, 'city'].unique()))
panel.head(3)

### Why we work in logs

A difference of logs is a **percentage change**. So the DiD estimate reads
directly as "+6%" rather than "+0.06 orders per active user", and large cities
stop dominating the average just because their absolute numbers are big.

`exp(x) - 1` converts a log difference back to a percentage.

In [ ]:
for col in ["orders_per_active", "gmv_per_active", "aov_inr", "session_conversion"]:
    panel["ln_" + col] = np.log(panel[col])

panel[["city", "week", "treated", "post", "orders_per_active", "ln_orders_per_active"]].head(3)

## 1. Adoption — did anyone actually use the feature?

Always check this first. If a feature shows no effect, you need to know whether
that means *"it doesn't help"* or *"nobody found the button"* — completely
different follow-ups.

In [ ]:
adopt = (panel[(panel.treated == 1) & (panel.post == 1)]
         .groupby("week").reorder_order_share.mean())

print(f"share of orders placed through the reorder flow:")
print(f"  first week {adopt.iloc[0]:.1%} | last week {adopt.iloc[-1]:.1%} | mean {adopt.mean():.1%}")

Roughly **28% of orders** in treated cities now go through the reorder flow, and
it doesn't decay. The feature is genuinely in use.

## 2. The naive answer — and why it's wrong

The tempting analysis: compare treated cities before vs after launch.

In [ ]:
t_pre  = panel[(panel.treated == 1) & (panel.post == 0)].orders_per_active.mean()
t_post = panel[(panel.treated == 1) & (panel.post == 1)].orders_per_active.mean()
naive  = t_post / t_pre - 1

c_pre  = panel[(panel.treated == 0) & (panel.post == 0)].orders_per_active.mean()
c_post = panel[(panel.treated == 0) & (panel.post == 1)].orders_per_active.mean()
control_drift = c_post / c_pre - 1

print(f"treated cities : {t_pre:.4f} -> {t_post:.4f}   = {naive:+.2%}   <- the tempting headline")
print(f"control cities : {c_pre:.4f} -> {c_post:.4f}   = {control_drift:+.2%}   <- no feature, still grew")
print(f"\nabout {control_drift / naive:.0%} of that headline is growth that happened anyway.")

The control cities never got the feature, yet they also grew — seasonality and
underlying growth. So the naive `+11%` is inflated. That gap is exactly what DiD
removes.

## 3. Parallel trends — am I even allowed to use DiD?

DiD assumes that, without the feature, the two groups would have **moved in
parallel** (their levels can differ — only the *slopes* must match).

Note the treated cities are the five **largest** ones — assignment was not
random, so this assumption deserves real scrutiny.

Test: fit each city's pre-launch trend, then compare the 5 treated slopes
against the 7 control slopes.

In [ ]:
def city_pre_slopes(ycol):
    rows = []
    pre = panel[panel.week < LAUNCH_WEEK]
    for city, g in pre.groupby("city"):
        rows.append({"city": city, "treated": g.treated.iloc[0],
                     "slope": np.polyfit(g.week, g[ycol], 1)[0]})
    return pd.DataFrame(rows)

def parallel_trends_test(ycol, label):
    s = city_pre_slopes(ycol)
    a = s.loc[s.treated == 1, "slope"]
    b = s.loc[s.treated == 0, "slope"]
    t, p = stats.ttest_ind(a, b, equal_var=False)
    verdict = "PASS - trends look parallel" if p > 0.10 else "FAIL - pre-trend!"
    print(f"{label:<22} treated {a.mean():+.5f}/wk  control {b.mean():+.5f}/wk  p={p:.3f}  {verdict}")

parallel_trends_test("ln_orders_per_active",  "orders/active")
parallel_trends_test("ln_gmv_per_active",     "GMV/active")
parallel_trends_test("ln_session_conversion", "conversion")
parallel_trends_test("cancel_rate",           "cancel rate")

The primary metrics pass. **Cancellation rate fails** — the treated cities were
already drifting upward before launch (a capacity problem, not a feature
problem). Any DiD number for cancellations is therefore *not* causal, and we
will refuse to report one. Catching this is the whole point of running the test.

Now look at it, because a chart is more convincing than a p-value.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
tr = panel[panel.treated == 1].groupby("week").orders_per_active.mean()
co = panel[panel.treated == 0].groupby("week").orders_per_active.mean()
ax.plot(tr.index, tr.values, lw=2.2, color="#2563eb", label="Treated (5 cities)")
ax.plot(co.index, co.values, lw=2.2, color="#94a3b8", label="Control (7 cities)")
ax.axvline(LAUNCH_WEEK - 0.5, color="#dc2626", ls="--", lw=1.4)
ax.text(LAUNCH_WEEK - 0.3, ax.get_ylim()[1]*0.99, "launch", color="#dc2626", va="top")
ax.set_xlabel("Week"); ax.set_ylabel("Orders per active user")
ax.set_title("Parallel before launch, diverging after", weight="bold")
ax.legend(frameon=False); ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
plt.show()

## 4. The DiD estimate

Four averages, two subtractions.

In [ ]:
def did_log(df, ycol, weeks=None):
    """DiD in logs. Pass `weeks` to restrict which POST weeks are used."""
    d = df if weeks is None else df[(df.week < LAUNCH_WEEK) | (df.week.isin(weeks))]
    m = d.groupby(["treated", "post"])[ycol].mean()
    return (m.loc[(1, 1)] - m.loc[(1, 0)]) - (m.loc[(0, 1)] - m.loc[(0, 0)])

steady_weeks = list(range(STEADY_FROM, panel.week.max() + 1))

eff_all    = np.exp(did_log(panel, "ln_orders_per_active")) - 1
eff_steady = np.exp(did_log(panel, "ln_orders_per_active", weeks=steady_weeks)) - 1

print(f"naive pre/post (wrong)      {naive:+.2%}")
print(f"DiD, whole post period      {eff_all:+.2%}")
print(f"DiD, steady state (W18+)    {eff_steady:+.2%}")

The real effect is roughly **half** the naive number.

Two DiD numbers are shown because they answer different questions: the full-post
figure includes the launch buzz, while the steady-state figure is what the
business will actually keep. Section 6 shows why they differ.

## 5. Could this be luck?

Only **12 cities** were involved. That is a small number of independent units,
and standard errors from a regression would be unreliable here. Two better
options:

**(a) Exact randomisation test.** There are only C(12,5) = 792 ways to pick 5
"treated" cities. Compute the DiD for *every* possible assignment. If the real
one stands out against all 792, it is unlikely to be chance.

In [ ]:
all_cities = sorted(panel.city.unique())
observed = did_log(panel, "ln_orders_per_active")

null_dist = []
for combo in combinations(all_cities, 5):
    fake = panel.copy()
    fake["treated"] = fake.city.isin(combo).astype(int)
    null_dist.append(did_log(fake, "ln_orders_per_active"))
null_dist = np.array(null_dist)

p_exact = (np.abs(null_dist) >= abs(observed)).mean()
print(f"assignments tested : {len(null_dist)}")
print(f"observed DiD (log) : {observed:+.4f}")
print(f"null range         : [{null_dist.min():+.4f}, {null_dist.max():+.4f}]")
print(f"two-sided p-value  : {p_exact:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4))
ax.hist(np.exp(null_dist) - 1, bins=40, color="#94a3b8")
ax.axvline(eff_all, color="#dc2626", lw=2.2, label=f"observed {eff_all:+.1%} (p={p_exact:.3f})")
ax.set_xlabel("DiD estimate under random re-assignment")
ax.set_ylabel("Count of the 792 assignments")
ax.set_title("The real estimate sits outside the placebo distribution", weight="bold")
ax.legend(frameon=False); ax.spines[["top","right"]].set_visible(False)
plt.show()

**(b) Cluster bootstrap for a confidence interval.** The thing that got treated
is a **city**, not a city-week. So resample whole cities with replacement
(keeping 5 treated and 7 control) and recompute the estimate many times.

In [ ]:
treated_pool = sorted(panel.loc[panel.treated == 1, "city"].unique())
control_pool = sorted(panel.loc[panel.treated == 0, "city"].unique())

def cluster_bootstrap_ci(ycol, weeks=None, reps=500):
    out = []
    for _ in range(reps):
        pick = (list(rng.choice(treated_pool, len(treated_pool), replace=True)) +
                list(rng.choice(control_pool, len(control_pool), replace=True)))
        parts = []
        for i, c in enumerate(pick):
            g = panel[panel.city == c].copy()
            g["city"] = f"{c}__{i}"          # keep duplicate draws distinct
            parts.append(g)
        out.append(did_log(pd.concat(parts, ignore_index=True), ycol, weeks))
    return np.exp(np.percentile(out, [2.5, 97.5])) - 1

lo, hi = cluster_bootstrap_ci("ln_orders_per_active", weeks=steady_weeks)
print(f"steady-state effect {eff_steady:+.2%}   95% CI [{lo:+.2%}, {hi:+.2%}]")

## 6. Event study — a launch spike or a real habit?

Plot the treated-vs-control gap **week by week**, recentred so the pre-launch
average gap is zero. Two things to read off it:

* the bars **before** launch should hover around zero — more evidence for
  parallel trends, one week at a time;
* the bars **after** launch show whether the effect holds up or fades.

In [ ]:
gap = panel.groupby(["week", "treated"]).ln_orders_per_active.mean().unstack()
gap["diff"] = gap[1] - gap[0]
baseline = gap.loc[gap.index < LAUNCH_WEEK, "diff"].mean()
gap["effect"] = np.exp(gap["diff"] - baseline) - 1
ev = gap["effect"]

fig, ax = plt.subplots(figsize=(9, 4.4))
rel = ev.index - LAUNCH_WEEK
ax.bar(rel, ev.values*100, width=0.75,
       color=["#94a3b8" if r < 0 else "#2563eb" for r in rel])
ax.axhline(0, color="#334155", lw=1)
ax.axvline(-0.5, color="#dc2626", ls="--", lw=1.4)
ax.axhline(eff_steady*100, color="#16a34a", ls=":", lw=1.6,
           label=f"steady state {eff_steady:+.1%}")
ax.set_xlabel("Weeks relative to launch"); ax.set_ylabel("Effect (%)")
ax.set_title("Launch spike decays to a durable effect within about a month", weight="bold")
ax.legend(frameon=False); ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
plt.show()

print(f"first 2 weeks {ev[[LAUNCH_WEEK, LAUNCH_WEEK+1]].mean():+.2%}"
      f"   |   week {STEADY_FROM}+ {ev[ev.index >= STEADY_FROM].mean():+.2%}")

This is why the launch-week number must never be the number you promise. Report
the steady state.

## 7. Secondary metrics and guardrails

A feature that lifts orders can still be bad for the business. Check money and
harm, not just the metric you hoped would move.

In [ ]:
for col, label in [("ln_orders_per_active", "orders per active"),
                   ("ln_gmv_per_active",    "GMV per active"),
                   ("ln_aov_inr",           "AOV"),
                   ("ln_session_conversion","session -> order conversion")]:
    e = np.exp(did_log(panel, col, weeks=steady_weeks)) - 1
    print(f"  {label:<28} {e:+.2%}")

print()
for col, label in [("cancel_rate", "cancellation rate (pp)"),
                   ("tickets_per_1k_orders", "support tickets / 1k orders")]:
    m = panel.groupby(["treated","post"])[col].mean()
    d = (m.loc[(1,1)] - m.loc[(1,0)]) - (m.loc[(0,1)] - m.loc[(0,0)])
    print(f"  {label:<28} {d:+.4f}")

**The important catch.** Orders are up ~6%, but **AOV is down ~2%** — reorder
baskets repeat staples and skip the browse-driven add-ons. Multiply them and GMV
per active user only rises about **+3.4%**, not 6%. Quoting the orders number
alone would oversell the feature by nearly half.

And the cancellation-rate number above must **not** be reported as an effect: it
failed the parallel-trends test in section 3. Here is what that failure looks
like.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
trc = panel[panel.treated == 1].groupby("week").cancel_rate.mean()*100
coc = panel[panel.treated == 0].groupby("week").cancel_rate.mean()*100
ax.plot(trc.index, trc.values, lw=2.2, color="#2563eb", label="Treated")
ax.plot(coc.index, coc.values, lw=2.2, color="#94a3b8", label="Control")
ax.axvline(LAUNCH_WEEK - 0.5, color="#dc2626", ls="--", lw=1.4)
ax.set_xlabel("Week"); ax.set_ylabel("Cancellation rate (%)")
ax.set_title("Already diverging BEFORE launch — so DiD cannot claim this one", weight="bold")
ax.legend(frameon=False); ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
plt.show()

## 8. Who does the feature actually help?

An average hides the decision. Split the user-level data by how habitual the
user already was, and run the same DiD within each segment.

In [ ]:
cs = (users.groupby(["city","treated","segment","period"])
      .agg(tot=("orders","sum"), n=("orders","size")).reset_index())

def segment_did(cities, segment):
    d = cs[cs.city.isin(cities) & (cs.segment == segment)]
    m = (d.groupby(["treated","period"])[["tot","n"]].sum()
         .assign(mean=lambda x: x.tot / x.n)["mean"])
    return (np.log(m.loc[(1,"post")]) - np.log(m.loc[(1,"pre")])
            - (np.log(m.loc[(0,"post")]) - np.log(m.loc[(0,"pre")])))

all_city_list = sorted(users.city.unique())
rows = []
for s in ["new", "occasional", "regular", "power"]:
    point = np.exp(segment_did(all_city_list, s)) - 1
    reps = []
    for _ in range(300):
        pick = (list(rng.choice(treated_pool, len(treated_pool), replace=True)) +
                list(rng.choice(control_pool, len(control_pool), replace=True)))
        reps.append(segment_did(list(set(pick)), s))
    lo_s, hi_s = np.exp(np.percentile(reps, [2.5, 97.5])) - 1
    rows.append({"segment": s, "effect": point, "ci_low": lo_s, "ci_high": hi_s})

het = pd.DataFrame(rows)
for _, r in het.iterrows():
    star = "" if r.ci_low < 0 < r.ci_high else "  *"
    print(f"  {r.segment:<12} {r.effect:+.2%}   95% CI [{r.ci_low:+.2%}, {r.ci_high:+.2%}]{star}")
print("\n  * = interval excludes zero")

The effect rises with existing habit. For **brand-new users the interval spans
zero** — no measurable benefit, which makes sense: a new user has no "last
order" worth repeating. That single finding changes the rollout plan from
"ship to everyone" to "ship to everyone, but stop counting on it to activate new
users."

## 9. Placebo test — does this method invent effects?

Pretend the launch happened at week 7 and analyse **only pre-launch data**.
Nothing happened then, so a trustworthy design should return roughly zero.

In [ ]:
pre_only = panel[panel.week < LAUNCH_WEEK].copy()
pre_only["post"] = (pre_only.week >= 7).astype(int)
placebo = np.exp(did_log(pre_only, "ln_orders_per_active")) - 1

print(f"placebo (fake launch, week 7) : {placebo:+.2%}   <- should be near zero")
print(f"real estimate                 : {eff_all:+.2%}")

## 10. What I would tell the product team

| Question | Answer |
|---|---|
| Did it work? | Yes — **+5.6% orders per active user** (95% CI +4.1% to +7.1%), p < 0.01 |
| How much is real money? | **+3.4% GMV per active**, because AOV fell ~2% |
| Is the launch number real? | No — the ~11% first fortnight is novelty; it settles near 6% |
| Any harm? | Support tickets flat. Cancellations can't be attributed (failed pre-trends) |
| Who benefits? | Habitual users (+9.5% power, +7.7% regular); no measurable effect for new users |

**Recommendation: roll out to all cities**, but

1. forecast with the steady-state number, not the launch spike;
2. track GMV per active rather than orders, since the basket shrinks;
3. pair it with a basket-building nudge (add-ons on the reorder screen) to
   recover the AOV loss;
4. don't expect it to help new-user activation — that needs a different feature;
5. open a **separate** investigation into the rising cancellations in the large
   cities, which pre-date this launch and are a capacity problem.

The full write-up is in `docs/decision_memo.md`; the method and its assumptions
are in `docs/methodology.md`.